In [1]:

"""
PIPELINE STAGE: Unstructured PDF-to-Numeric Environmental Covariate Synthesis
SOURCE: D:\dergi\raw_data\daylight\*.pdf (Comprehensive Annual Solar Calendars)
OUTPUT: D:\dergi\processed_data\steps\12_daylight.xlsx
LIBRARIES: pdfplumber, pandas, os, re

1. RESEARCH OBJECTIVE: GEOSPATIAL PHOTOPERIOD EXTRACTION
    The goal is to synthesize a high-fidelity environmental dataset from 81 independent 
    annual solar calendars. By extracting Sunrise and Sunset markers, the script 
    engineers the 'Daylight Duration' feature, which serves as a proxy for 
    lighting-related residential electricity demand in spatial energy models.

2. METHODOLOGICAL ARCHITECTURE:
    - Representative Sampling: Instead of processing 365 daily rows, the algorithm 
      isolates the 15th day of each month. This serves as a climatological midpoint, 
      aligning perfectly with longitudinal monthly energy consumption data.
    - Time-Arithmetic Normalization: Raw HH:MM timestamps (e.g., 05:42 and 19:15) 
      are converted into total minutes and سپس transformed into numeric decimal hours 
      (e.g., 13.55). This allows for linear regression and correlation analysis.
    - Relational Key Mapping: Categorical month names (Ocak, Şubat...) and province 
      names are mapped to unique integers (Month: 1-12, Plate: 1-81) to ensure 
      relational integrity across the master database.

3. DATA TRANSFORMATION LOGIC:
    - PDF Scraping: Utilizing a fallback regex engine to handle unstructured text 
      layout across multi-page Diyanet PDF documents.
    - Localization: Floating-point durations are formatted with comma separators 
      (e.g., 10,50) to comply with regional (Turkish) spreadsheet standards.
    - Hierarchical Sorting: The final dataset is multi-level sorted by [Plate Index] 
      and [Month ID] in ascending order for seamless panel-data integration.

4. ACADEMIC VALUE:
    This stage transforms qualitative religious calendars into quantitative 
    climatological intelligence, reducing dimensionality while maintaining 
    geospatial precision for 81 administrative units.
"""


import pdfplumber
import pandas as pd
import os
import re

# Klasör ve Dosya yolları
kaynak_dizin = os.path.join("..", "..", "raw_data", "daylight")
cikis_dosyasi = os.path.join("..", "..", "processed_data", "steps","12_daylight.xlsx")


# Plaka Map (İl isimlerini plakaya bağlar)
plate_map = {
    "Adana": 1, "Adıyaman": 2, "Afyon": 3, "Afyonkarahisar": 3, "Ağrı": 4, "Amasya": 5, "Ankara": 6, "Antalya": 7, "Artvin": 8, "Aydın": 9,
    "Balıkesir": 10, "Bilecik": 11, "Bingöl": 12, "Bitlis": 13, "Bolu": 14, "Burdur": 15, "Bursa": 16, "Çanakkale": 17, "Çankırı": 18, "Çorum": 19,
    "Denizli": 20, "Diyarbakır": 21, "Edirne": 22, "Elazığ": 23, "Erzincan": 24, "Erzurum": 25, "Eskişehir": 26, "Gaziantep": 27, "Giresun": 28, "Gümüşhane": 29,
    "Hakkari": 30, "Hatay": 31, "Isparta": 32, "Mersin": 33, "İçel": 33, "İstanbul": 34, "İzmir": 35, "Kars": 36, "Kastamonu": 37, "Kayseri": 38, "Kırklareli": 39,
    "Kırşehir": 40, "Kocaeli": 41, "Konya": 42, "Kütahya": 43, "Malatya": 44, "Manisa": 45, "Kahramanmaraş": 46, "Mardin": 47, "Muğla": 48, "Muş": 49,
    "Nevşehir": 50, "Niğde": 51, "Ordu": 52, "Rize": 53, "Sakarya": 54, "Samsun": 55, "Siirt": 56, "Sinop": 57, "Sivas": 58, "Tekirdağ": 59,
    "Tokat": 60, "Trabzon": 61, "Tunceli": 62, "Şanlıurfa": 63, "Uşak": 64, "Van": 65, "Yozgat": 66, "Zonguldak": 67, "Aksaray": 68, "Bayburt": 69,
    "Karaman": 70, "Kırıkkale": 71, "Batman": 72, "Şırnak": 73, "Bartın": 74, "Ardahan": 75, "Iğdır": 76, "Yalova": 77, "Karabük": 78, "Kilis": 79,
    "Osmaniye": 80, "Düzce": 81
}

# Ay Map (Ay isimlerini numaraya bağlar)
month_map = {
    "Ocak": 1, "Şubat": 2, "Mart": 3, "Nisan": 4, "Mayıs": 5, "Haziran": 6,
    "Temmuz": 7, "Ağustos": 8, "Eylül": 9, "Ekim": 10, "Kasım": 11, "Aralık": 12
}

def saat_farki_hesapla_virgullu(gunes_str, aksam_str):
    try:
        sh, sm = map(int, gunes_str.split(':'))
        ah, am = map(int, aksam_str.split(':'))
        baslangic_dk = (sh * 60) + sm
        bitis_dk = (ah * 60) + am
        fark_saat = (bitis_dk - baslangic_dk) / 60
        return str(round(fark_saat, 2)).replace('.', ',')
    except:
        return "0,0"

def veri_ayikla_numerik_final():
    tum_illerin_verisi = []
    saat_deseni = re.compile(r'\d{2}:\d{2}')
    pdf_dosyalari = [f for f in os.listdir(kaynak_dizin) if f.endswith(".pdf")]
    
    for dosya_adi in pdf_dosyalari:
        il_adi_temiz = dosya_adi.split()[0]
        plate_no = plate_map.get(il_adi_temiz, 0)
        
        if plate_no == 0: continue

        print(f"Processing: {il_adi_temiz} (Plate {plate_no})...")
        pdf_yolu = os.path.join(kaynak_dizin, dosya_adi)
        
        try:
            with pdfplumber.open(pdf_yolu) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if not text: continue
                    
                    for line in text.split('\n'):
                        # Ay ismini bul
                        bulunan_ay = next((ay for ay in month_map.keys() if ay in line), None)
                        
                        if bulunan_ay and len(saat_deseni.findall(line)) >= 5:
                            parcalar = line.split()
                            if parcalar[0] == "15":
                                vakitler = saat_deseni.findall(line)
                                # Ay ismini numaraya çevir
                                month_no = month_map[bulunan_ay]
                                daylight_val = saat_farki_hesapla_virgullu(vakitler[1], vakitler[4])

                                tum_illerin_verisi.append([
                                    plate_no,
                                    il_adi_temiz,
                                    month_no,
                                    daylight_val
                                ])
        except Exception as e:
            print(f"Error at Plate {plate_no}: {e}")

    # DataFrame Oluşturma
    columns = ["Plate", "Province", "Month", "Daylight"]
    df = pd.DataFrame(tum_illerin_verisi, columns=columns).drop_duplicates()
    
    # 1. PLAKA ARTAN SIRADA, 2. AY NUMARASI ARTAN SIRADA
    df = df.sort_values(by=["Plate", "Month"]).reset_index(drop=True)

   
    
    df.to_excel(cikis_dosyasi, index=False)
    
    print("\n" + "="*50)
    print("SIRALAMA VE DÖNÜŞTÜRME TAMAMLANDI")
    print(f"Örnek (İlk 12 Satır):\n{df.head(12).to_string(index=False)}")
    print(f"Kayıt: {cikis_dosyasi}")
    print("="*50)

if __name__ == "__main__":
    veri_ayikla_numerik_final()

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ismailgulsoy\AppData\Local\Temp\ipykernel_5736\3411551322.py:1: SyntaxWarning: invalid escape sequence '\d'
  """


Processing: Adana (Plate 1)...
Processing: Adıyaman (Plate 2)...
Processing: Afyonkarahisar (Plate 3)...
Processing: Aksaray (Plate 68)...
Processing: Amasya (Plate 5)...
Processing: Ankara (Plate 6)...
Processing: Antalya (Plate 7)...
Processing: Ardahan (Plate 75)...
Processing: Artvin (Plate 8)...
Processing: Aydın (Plate 9)...
Processing: Ağrı (Plate 4)...
Processing: Balıkesir (Plate 10)...
Processing: Bartın (Plate 74)...
Processing: Batman (Plate 72)...
Processing: Bayburt (Plate 69)...
Processing: Bilecik (Plate 11)...
Processing: Bingöl (Plate 12)...
Processing: Bitlis (Plate 13)...
Processing: Bolu (Plate 14)...
Processing: Burdur (Plate 15)...
Processing: Bursa (Plate 16)...
Processing: Denizli (Plate 20)...
Processing: Diyarbakır (Plate 21)...
Processing: Düzce (Plate 81)...
Processing: Edirne (Plate 22)...
Processing: Elazığ (Plate 23)...
Processing: Erzincan (Plate 24)...
Processing: Erzurum (Plate 25)...
Processing: Eskişehir (Plate 26)...
Processing: Gaziantep (Plate 27